# DistilBERT bi-encoder: training

Each address is encoded alone (`[CLS] A [SEP]`, at most 64 word pieces), masked mean pooling, head on `[|u−v|; u⊙v]`: the siamese BiLSTM's set-up with DistilBERT (`distilbert-base-uncased`) as the encoder.

Protocol (`training.py`): AdamW, encoder 2e-5 / head 1e-3, weight decay 0.01, batch 32, linear warm-up over the first 10% of steps then decay, at most 5 epochs, patience 2; best epoch by the shared rule (≥ 98% of val case 1 rejected, then most val non-identical matches found). Seeds 0, 1, 2. The paper's runs were trained on a Google Colab T4 GPU (about 10 min per epoch).

Writes `checkpoints/distilbert_bi/` (git-ignored; `seed_*_last.pt` lets a disconnected run resume) and `predictions/distilbert_bi/` (history + val/test probabilities).

In [ ]:
# Colab only (does nothing locally): mount Google Drive and work from a copy of this repository there,
# so checkpoints survive a disconnect. Edit REPO_ON_DRIVE to where you put it.
import os
REPO_ON_DRIVE = "/content/drive/MyDrive/address-matching"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir(os.path.join(REPO_ON_DRIVE, "distilbert"))
    print("Colab: working in", os.getcwd())
except ImportError:
    pass

In [ ]:
import os
import sys
sys.path.append("..")   # repository root -> lets us import helpers/ and the model packages

import pandas as pd

from helpers.pairs import load_pairs
from distilbert.model import make_loaders
from distilbert.training import run_training, save_predictions, checkpoint_dir, SEEDS, DEVICE
from siamese.training import broken_run_problems

KIND = "bi"
DATASET = "overture_case2s"

train, train_label = load_pairs(DATASET, "train")
val, val_label = load_pairs(DATASET, "val")
test, test_label = load_pairs(DATASET, "test")
loaders = make_loaders(KIND, {"train": (train, train_label), "val": (val, val_label), "test": (test, test_label)})
DEVICE

## 1. Train each seed (skipped once `seed_*_final.pt` exists)

In [ ]:
for seed in SEEDS:
    if os.path.exists(f"{checkpoint_dir(KIND)}/seed_{seed}_final.pt"):
        print(f"seed {seed}: already finished, skipping")
        continue
    history, best, best_state = run_training(KIND, seed, loaders, val, val_label)
    print(seed, broken_run_problems(history))

## 2. Val / test probabilities and history → `predictions/`

In [ ]:
for seed in SEEDS:
    save_predictions(KIND, seed, loaders)
    h = pd.read_csv(f"../predictions/distilbert_{KIND}/history_seed_{seed}.csv")
    print(f"seed {seed}: problems {broken_run_problems(h)}, epochs run {len(h)}")